# Block masking parameter exploration

Loads real APA sparse images, applies `SparseBlockMasker` (the same class used during training),
and visualises which voxels get masked.  
Use this to calibrate `seed_frac`, `win_ch`, and `win_tick` before training.

`SparseBlockMasker` removes masked voxels and returns their coordinates.  
The helpers below reconstruct a `mask_bool` from the original coords and the removed-coord list
so we can colour kept vs. masked voxels in the scatter plots.

In [ ]:
import sys
sys.path.insert(0, "/nfs/data/1/mvicenzi/ml-dune-model")

import math
import torch
import numpy as np
import matplotlib.pyplot as plt

from loader.apa_sparse_dataset import APASparseDataset
from loader.collate import voxels_collate_fn
from torch.utils.data import DataLoader

from dino.masking import SparseBlockMasker

## Load a small batch

In [ ]:
DATADIR   = "/nfs/data/1/yuhw/cffm-data/prod-jay-100k-truth-2026-02-27"
CACHE_DIR = "/nfs/data/1/mvicenzi/ml-dune-model/data"
DEVICE    = "cpu"   # change to "cuda" if a GPU is available

ds = APASparseDataset(
    datadir=DATADIR,
    apa=0,
    view="W",
    use_cache=True,
    cache_dir=CACHE_DIR,
)
print("Dataset size:", len(ds))

SAMPLE_INDICES = [0, 1, 2, 3]
N_BATCH = len(SAMPLE_INDICES)

loader = DataLoader(
    torch.utils.data.Subset(ds, SAMPLE_INDICES),
    batch_size=N_BATCH,
    shuffle=False,
    num_workers=0,
    collate_fn=voxels_collate_fn,
)
vox = next(iter(loader)).to(DEVICE)

print("Coordinates shape :", vox.coordinate_tensor.shape)
print("Features shape    :", vox.feature_tensor.shape)
print("Offsets           :", vox.offsets.tolist())

counts = (vox.offsets[1:] - vox.offsets[:-1]).tolist()
print("Active voxels per image:", [int(c) for c in counts])

## Helpers

In [ ]:
def reconstruct_mask_bool(orig_coords, masked_coords):
    """
    Given the original voxel coordinates and the list of removed coordinates,
    return a bool tensor (shape N_orig) that is True at masked positions.
    Uses flat keys (tick * W + channel) for O(N log N) lookup.
    """
    if orig_coords.shape[0] == 0:
        return torch.zeros(0, dtype=torch.bool)
    W = int(orig_coords[:, 0].max().item()) + 1
    orig_keys   = orig_coords[:, 1].long() * W + orig_coords[:, 0].long()
    if masked_coords.shape[0] == 0:
        return torch.zeros(orig_coords.shape[0], dtype=torch.bool)
    masked_keys = masked_coords[:, 1].long() * W + masked_coords[:, 0].long()
    return torch.isin(orig_keys, masked_keys)


def apply_and_split(vox, masker):
    """
    Run masker on the batch and return per-image (coords, feats, mask_bool) tuples.
    coords and feats come from the *original* vox so masked voxels are still visible.
    """
    _, masked_coords_per_batch = masker(vox)
    per_image = []
    B = len(vox.offsets) - 1
    for b in range(B):
        start    = int(vox.offsets[b])
        end      = int(vox.offsets[b + 1])
        coords_b = vox.coordinate_tensor[start:end]
        feats_b  = vox.feature_tensor[start:end, 0]
        mask_b   = reconstruct_mask_bool(coords_b, masked_coords_per_batch[b])
        per_image.append((coords_b, feats_b, mask_b))
    return per_image


def plot_masked(ax, coords, feats, mask_bool, title="", image_h=1500, image_w=1050):
    """
    Scatter-plot one image.  Kept voxels are coloured by charge (viridis);
    masked voxels are shown in red.
    """
    coords_np = coords.cpu().numpy()
    feats_np  = feats.cpu().numpy()
    mask_np   = mask_bool.cpu().numpy()
    kept      = ~mask_np
    n_masked  = mask_np.sum()
    frac      = n_masked / len(mask_np) if len(mask_np) > 0 else 0.0

    if kept.sum() > 0:
        ax.scatter(coords_np[kept, 0], coords_np[kept, 1],
                   c=feats_np[kept], cmap="viridis", s=1, linewidths=0, alpha=0.8)
    if n_masked > 0:
        ax.scatter(coords_np[mask_np, 0], coords_np[mask_np, 1],
                   c="red", s=2, linewidths=0, alpha=0.6, label="masked")

    ax.set_xlim(0, image_w)
    ax.set_ylim(0, image_h)
    ax.invert_yaxis()
    ax.set_xlabel("Channel")
    ax.set_ylabel("Tick")
    ax.set_title(f"{title}\nmasked {n_masked}/{len(mask_np)} ({frac:.1%})", fontsize=8)

## Vary `seed_frac` at fixed window

Each column is one value of `seed_frac`.  Red = masked, viridis = kept.

In [ ]:
IMAGE_IDX  = 0
WIN_CH     = 5
WIN_TICK   = 5
SEED_FRACS = [0.02, 0.05, 0.10, 0.20, 0.40]

fig, axes = plt.subplots(1, len(SEED_FRACS), figsize=(4 * len(SEED_FRACS), 5))

for ax, sf in zip(axes, SEED_FRACS):
    masker    = SparseBlockMasker(seed_frac=sf, win_ch=WIN_CH, win_tick=WIN_TICK)
    per_image = apply_and_split(vox, masker)
    coords_b, feats_b, mask_b = per_image[IMAGE_IDX]
    plot_masked(ax, coords_b, feats_b, mask_b,
                title=f"seed_frac={sf:.2f}\nwin=({WIN_CH},{WIN_TICK})")

plt.suptitle(f"Block masking — image {IMAGE_IDX} — varying seed_frac", fontsize=11)
plt.tight_layout()
plt.show()

## Window size grid: vary `(win_ch, win_tick)` at fixed `seed_frac`

Rows = `win_ch`, columns = `win_tick`.

In [ ]:
IMAGE_IDX       = 0
SEED_FRAC       = 0.05
WIN_CH_VALUES   = [2, 5, 10, 20]
WIN_TICK_VALUES = [2, 5, 10, 20]

nrows = len(WIN_CH_VALUES)
ncols = len(WIN_TICK_VALUES)
fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 4 * nrows))

for i, wc in enumerate(WIN_CH_VALUES):
    for j, wt in enumerate(WIN_TICK_VALUES):
        masker    = SparseBlockMasker(seed_frac=SEED_FRAC, win_ch=wc, win_tick=wt)
        per_image = apply_and_split(vox, masker)
        coords_b, feats_b, mask_b = per_image[IMAGE_IDX]
        plot_masked(axes[i, j], coords_b, feats_b, mask_b,
                    title=f"win_ch={wc}, win_tick={wt}")

plt.suptitle(f"Window grid — image {IMAGE_IDX} — seed_frac={SEED_FRAC}", fontsize=11)
plt.tight_layout()
plt.show()

## Masked-fraction statistics

Plot actual masked fraction vs `seed_frac` for several window configurations.  
The dashed diagonal is the pixel-by-pixel reference (masked fraction = seed_frac).  
Block masking always lies above it because each seed expands a window.

In [ ]:
SEED_FRACS_SWEEP = np.linspace(0.01, 0.50, 20)
WINDOW_CONFIGS   = [(2, 2), (5, 5), (10, 10), (20, 20), (5, 20)]

total_voxels = vox.feature_tensor.shape[0]
results = {}  # (win_ch, win_tick) -> list of mean masked fractions

for win_ch, win_tick in WINDOW_CONFIGS:
    fracs = []
    for sf in SEED_FRACS_SWEEP:
        masker = SparseBlockMasker(seed_frac=float(sf), win_ch=win_ch, win_tick=win_tick)
        _, masked_coords_per_batch = masker(vox)
        n_masked = sum(m.shape[0] for m in masked_coords_per_batch)
        fracs.append(n_masked / total_voxels)
    results[(win_ch, win_tick)] = fracs

fig, ax = plt.subplots(figsize=(8, 5))
for (wc, wt), fracs in results.items():
    ax.plot(SEED_FRACS_SWEEP, fracs, marker=".", label=f"win=({wc},{wt})")
ax.plot([0, 0.5], [0, 0.5], "k--", linewidth=0.8, label="pixel-by-pixel reference")

ax.set_xlabel("seed_frac")
ax.set_ylabel("actual masked fraction")
ax.set_title("seed_frac → actual masked fraction  (averaged over batch)")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## All images in batch — chosen parameters

Confirm the chosen parameters look sensible across different events.

In [ ]:
SEED_FRAC = 0.05
WIN_CH    = 5
WIN_TICK  = 5

masker    = SparseBlockMasker(seed_frac=SEED_FRAC, win_ch=WIN_CH, win_tick=WIN_TICK)
per_image = apply_and_split(vox, masker)

fig, axes = plt.subplots(1, N_BATCH, figsize=(4 * N_BATCH, 5))
if N_BATCH == 1:
    axes = [axes]

for b, (ax, (coords_b, feats_b, mask_b)) in enumerate(zip(axes, per_image)):
    plot_masked(ax, coords_b, feats_b, mask_b, title=f"image {SAMPLE_INDICES[b]}")

plt.suptitle(
    f"Block masking — seed_frac={SEED_FRAC}, win_ch={WIN_CH}, win_tick={WIN_TICK}",
    fontsize=11,
)
plt.tight_layout()
plt.show()

## Block shape: zoom into one masked region

Pick one seed at random and draw its expanded window to see the actual block shape on the sparse data.

In [ ]:
from matplotlib.patches import Rectangle

IMAGE_IDX = 0
WIN_CH    = 5
WIN_TICK  = 5
ZOOM_PAD  = 20

start    = int(vox.offsets[IMAGE_IDX])
end      = int(vox.offsets[IMAGE_IDX + 1])
coords_b = vox.coordinate_tensor[start:end]   # (N, 2)
feats_b  = vox.feature_tensor[start:end, 0]   # (N,)
N        = coords_b.shape[0]

seed_idx   = torch.randint(0, N, (1,)).item()
seed_coord = coords_b[seed_idx]
sc, st     = int(seed_coord[0]), int(seed_coord[1])

diff   = coords_b - seed_coord.unsqueeze(0)
in_win = (diff[:, 0].abs() <= WIN_CH) & (diff[:, 1].abs() <= WIN_TICK)

z_ch_lo = sc - WIN_CH  - ZOOM_PAD
z_ch_hi = sc + WIN_CH  + ZOOM_PAD
z_tk_lo = st - WIN_TICK - ZOOM_PAD
z_tk_hi = st + WIN_TICK + ZOOM_PAD
in_zoom = ((coords_b[:, 0] >= z_ch_lo) & (coords_b[:, 0] <= z_ch_hi) &
            (coords_b[:, 1] >= z_tk_lo) & (coords_b[:, 1] <= z_tk_hi))

coords_np  = coords_b.cpu().numpy()
feats_np   = feats_b.cpu().numpy()
in_win_np  = in_win.cpu().numpy()
in_zoom_np = in_zoom.cpu().numpy()

kept   = in_zoom_np & ~in_win_np
masked = in_zoom_np &  in_win_np

fig, ax = plt.subplots(figsize=(6, 6))
if kept.sum() > 0:
    ax.scatter(coords_np[kept, 0], coords_np[kept, 1],
               c=feats_np[kept], cmap="viridis", s=10, linewidths=0, alpha=0.8)
if masked.sum() > 0:
    ax.scatter(coords_np[masked, 0], coords_np[masked, 1],
               c="red", s=15, linewidths=0, alpha=0.9, label="masked (window)")

ax.scatter([sc], [st], marker="*", s=200, c="gold", zorder=5, label="seed")
rect = Rectangle(
    (sc - WIN_CH - 0.5, st - WIN_TICK - 0.5),
    2 * WIN_CH + 1, 2 * WIN_TICK + 1,
    linewidth=1.5, edgecolor="orange", facecolor="none", linestyle="--",
)
ax.add_patch(rect)

ax.set_xlim(z_ch_lo, z_ch_hi)
ax.set_ylim(z_tk_lo, z_tk_hi)
ax.invert_yaxis()
ax.set_xlabel("Channel")
ax.set_ylabel("Tick")
ax.set_title(f"Zoom: seed at (ch={sc}, tick={st})\nwin_ch={WIN_CH}, win_tick={WIN_TICK}")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()